# 🤖 Notebook 03 – ML Model Training & Comparison
**Project:** Student Performance Prediction Using ML
**Intern:** SEERAPU SRAVANI | 24U45A4219 | CSE-AIML, JNTUGV

Models Covered: Logistic Regression, Decision Tree, Random Forest, SVM, KNN, Gradient Boosting, XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load and Prepare Data

In [ ]:
df = pd.read_csv('../data/student_data.csv')
df = df.drop(columns=['StudentID','FinalMarks','FinalGrade'])
le = LabelEncoder()
for col in ['Gender','SocioeconomicStatus','ParentEducation']:
    df[col] = le.fit_transform(df[col])
X = df.drop(columns=['Result'])
y = le.fit_transform(df['Result'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Data ready. Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 2. Train All Models

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)'           : SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'KNN (K=7)'           : KNeighborsClassifier(n_neighbors=7),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=200, learning_rate=0.05,
                                          max_depth=6, use_label_encoder=False,
                                          eval_metric='logloss', random_state=42),
}

results = {}
print(f'{"Model":<22} {"Accuracy":>10} {"AUC":>8} {"10-CV":>10}')
print('-'*55)
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    yp = mdl.predict(X_test)
    yproba = mdl.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, yp)
    auc = roc_auc_score(y_test, yproba)
    cv  = cross_val_score(mdl, X_scaled, y, cv=10, scoring='accuracy').mean()
    results[name] = {'accuracy':acc,'auc':auc,'cv':cv,'model':mdl,'pred':yp,'proba':yproba}
    print(f'{name:<22} {acc*100:>9.2f}%  {auc:>7.4f}  {cv*100:>9.2f}%')

## 3. Model Comparison Chart

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy']*100 for n in names]
aucs  = [results[n]['auc'] for n in names]
best  = names[np.argmax(accs)]
colors = ['#d62728' if n==best else '#2E74B5' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

bars = axes[0].barh(names, accs, color=colors, edgecolor='white', height=0.6)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Test Accuracy')
axes[0].set_xlim(50, 100)
for b, v in zip(bars, accs):
    axes[0].text(v+0.3, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)

bars2 = axes[1].barh(names, aucs, color=colors, edgecolor='white', height=0.6)
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('AUC Score')
axes[1].set_xlim(0.5, 1.0)
for b, v in zip(bars2, aucs):
    axes[1].text(v+0.002, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best model: {best}')

## 4. Confusion Matrices (All Models)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flat
fig.suptitle('Confusion Matrices – All Models', fontsize=14, fontweight='bold')
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Fail','Pass'], yticklabels=['Fail','Pass'],
                cbar=False, linewidths=0.5)
    ax.set_title(f'{name}\n({res["accuracy"]*100:.1f}%)', fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Best Model – Detailed Report

In [ ]:
best_res = results[best]
print(f'Best Model: {best}')
print(f'Accuracy  : {best_res["accuracy"]*100:.2f}%')
print(f'AUC       : {best_res["auc"]:.4f}')
print(f'10-Fold CV: {best_res["cv"]*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, best_res['pred'], target_names=['Fail','Pass']))

## ✅ Model Training Complete
- 7 models trained and evaluated
- XGBoost achieved the best performance
- Proceed to Notebook 04 for hyperparameter tuning